# **Model1 : RandomForest/ Lgbm/ GradientBoosting Ensemble**


## 1. Libraries

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from string import ascii_lowercase
from itertools import combinations

import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import  GradientBoostingClassifier

from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score

## 2. Loading the data


In [2]:
train = pd.read_csv('./data/train.csv')
test = pd.read_csv('./data/test_x.csv')

## 4. Feature Engineering

In [3]:
x_train = train.copy()
x_train.drop('voted', axis=1, inplace = True)
y_train = train['voted']

In [4]:
dataset = [x_train, test]

### 마키아밸리 테스트 FE

In [5]:
questions = [i for i in list(ascii_lowercase)[:20]]
answers = [('Q'+i+'A') for i in questions]

In [6]:
for data in dataset:
  data['T'] = data['QcA'] - data['QfA'] + data['QoA'] - data['QrA'] + data['QsA']
  data['V'] = data['QbA'] - data['QeA'] + data['QhA'] + data['QjA'] + data['QmA'] - data['QqA']
  # data['M'] = - data['QkA']

Tactic/ Morality/ View에 따라 feature 항목을 나눠보았습니다.

In [7]:
flipping_columns = ["QeA", "QfA", "QkA", "QqA", "QrA"]
for data in dataset:
  for flip in flipping_columns: 
    data[flip] = 6 - data[flip]

In [8]:
flipping_secret_columns = ["QaA", "QdA", "QgA", "QiA", "QnA"]
for data in dataset:
  for flip in flipping_secret_columns: 
    data[flip] = 6 - data[flip]

In [9]:
for data in dataset:
  data['Mach_score'] = data[answers].mean(axis = 1)

In [10]:
for data in dataset:
  data['delay'] = data[[('Q'+i+'E') for i in questions]].sum(axis=1)
  data['delay'] = data['delay'] ** (1/10)

In [11]:
Ancoms = list(combinations(answers, 2))
for data in dataset:
  for a,b in Ancoms:
    data['%s_dv_%s'%(a,b)] = data[a]/data[b]

/tmp/ipykernel_810765/1581226005.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data['%s_dv_%s'%(a,b)] = data[a]/data[b]
/tmp/ipykernel_810765/1581226005.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data['%s_dv_%s'%(a,b)] = data[a]/data[b]
/tmp/ipykernel_810765/1581226005.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented fra

In [12]:
for data in dataset:
  data.drop([('Q'+i+'A') for i in questions], axis = 1, inplace = True)
  data.drop([('Q'+i+'E') for i in questions], axis = 1, inplace = True)

### 나머지 Features


In [13]:
for data in dataset:
  data.drop('hand', axis=1, inplace = True)

In [14]:
wr_list = [('wr_0'+str(i)) for i in range(1,10)]
wr_list.extend([('wr_'+str(i)) for i in range(10,14)])
wr_no_need = [i for i in wr_list if i not in ['wr_01', 'wr_03', 'wr_06', 'wr_09', 'wr_11']]

EDA에서 결과에 큰 영향이 없다고 판단된 feature들을 제거해주었습니다.

In [15]:
for data in dataset:
  data.drop(wr_no_need, axis=1, inplace = True)

In [16]:
for data in dataset:
  data['Ex'] = (data['tp01']+data['tp06'])/2
  data['Ag'] = (data['tp07']+data['tp02'])/2
  data['Con'] = (data['tp03']+data['tp08'])/2
  data['Es'] =(data['tp09']+data['tp04'])/2
  data['Op'] =(data['tp05']+data['tp10'])/2

TIPI test에 따라 feature 항목을 나눠놓았는데, 이때는 tipi feature들이 flip된 형태로 저장되어있는지 몰라서 따로 전처리를 해주지 않았었습니다.

In [17]:
for data in dataset:
  data.drop([('tp0'+str(i)) for i in range(1,10)], axis=1, inplace = True)
  data.drop('tp10', axis = 1, inplace = True)

In [18]:
index = test['index']
for data in dataset:
  data.drop('index', axis = 1, inplace = True)

In [19]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
needenco = ['age_group', 'gender', 'race', 'religion']
for i in needenco:
  x_train[i] = encoder.fit_transform(x_train[i])
  test[i] = encoder.transform(test[i])

## 5. Model

In [20]:
k_fold = KFold(n_splits = 3, shuffle = True, random_state = 0)

In [21]:
from catboost import CatBoostClassifier

# clf1 = RandomForestClassifier(n_estimators=500)
params = {'bagging_temperature': 0.375906, 'depth': 9.0, 'l2_leaf_reg': 68.8, 'learning_rate': 0.011, 'subsample': 0.76046}

clf1 = CatBoostClassifier(**params, iterations=5000, eval_metric='AUC',allow_writing_files=False,od_type='Iter',task_type='GPU',random_state=777)
# clf1 = CatBoostClassifier(learning_rate=0.05, iterations=5000, task_type="GPU")
clf2 = LGBMClassifier()
clf3 = GradientBoostingClassifier()
soft_vote  = VotingClassifier([('r1',clf1), ('r2', clf2), ('r3',clf3)], voting='soft')
soft_vote.fit(x_train, y_train)

0:	total: 80.4ms	remaining: 6m 41s
1:	total: 111ms	remaining: 4m 38s
2:	total: 139ms	remaining: 3m 52s
3:	total: 169ms	remaining: 3m 31s
4:	total: 199ms	remaining: 3m 19s
5:	total: 231ms	remaining: 3m 12s
6:	total: 262ms	remaining: 3m 7s
7:	total: 291ms	remaining: 3m 1s
8:	total: 318ms	remaining: 2m 56s
9:	total: 343ms	remaining: 2m 51s
10:	total: 368ms	remaining: 2m 46s
11:	total: 393ms	remaining: 2m 43s
12:	total: 417ms	remaining: 2m 39s
13:	total: 440ms	remaining: 2m 36s
14:	total: 464ms	remaining: 2m 34s
15:	total: 488ms	remaining: 2m 31s
16:	total: 511ms	remaining: 2m 29s
17:	total: 534ms	remaining: 2m 27s
18:	total: 557ms	remaining: 2m 26s
19:	total: 579ms	remaining: 2m 24s
20:	total: 602ms	remaining: 2m 22s
21:	total: 623ms	remaining: 2m 21s
22:	total: 646ms	remaining: 2m 19s
23:	total: 667ms	remaining: 2m 18s
24:	total: 690ms	remaining: 2m 17s
25:	total: 711ms	remaining: 2m 16s
26:	total: 734ms	remaining: 2m 15s
27:	total: 755ms	remaining: 2m 14s
28:	total: 777ms	remaining: 2m 

VotingClassifier(estimators=[('r1',
                              <catboost.core.CatBoostClassifier object at 0x7fb8d467fcd0>),
                             ('r2', LGBMClassifier()),
                             ('r3', GradientBoostingClassifier())],
                 voting='soft')

In [22]:
model = soft_vote
pred_y = model.predict_proba(test)
pred_y = pred_y[:,1]

submission = pd.DataFrame({
    "index" : index,
    "voted" : pred_y
})
submission.to_csv('./data/model1_2.csv', index=False)


# **Model2: Lgbm Ensemble with different features**

## 1. Libraries

In [23]:
import pandas as pd
import numpy as np

from string import ascii_lowercase
from itertools import combinations

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score

In [24]:
from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestRegressor
import eli5
from eli5.sklearn import PermutationImportance

import matplotlib.pyplot as plt

import warnings
import gc
warnings.filterwarnings("ignore")

## 2. Loading the data

In [25]:
train = pd.read_csv('./data/train.csv')
test = pd.read_csv('./data/test_x.csv')

## 3. Feature Engineering

In [26]:
x_train = train.copy()
x_train.drop('voted', axis=1, inplace = True)
y_train = train['voted']

In [27]:
dataset = [x_train, test]





### 마키아밸리 테스트 FE

In [28]:
questions = [i for i in list(ascii_lowercase)[:20]]
answers = [('Q'+i+'A') for i in questions]

In [29]:
for data in dataset:
  data['T'] = data['QcA'] - data['QfA'] + data['QoA'] - data['QrA'] + data['QsA']
  data['V'] = data['QbA'] - data['QeA'] + data['QhA'] + data['QjA'] + data['QmA'] - data['QqA']
  # data['M'] = - data['QkA']

In [30]:
flipping_columns = ["QeA", "QfA", "QkA", "QqA", "QrA"]
for data in dataset:
  for flip in flipping_columns: 
    data[flip] = 6 - data[flip]

In [31]:
flipping_secret_columns = ["QaA", "QdA", "QgA", "QiA", "QnA"]
for data in dataset:
  for flip in flipping_secret_columns: 
    data[flip] = 6 - data[flip]

In [32]:
for data in dataset:
  data['Mach_score'] = data[answers].mean(axis = 1)

In [33]:
for data in dataset:
  data['delay'] = data[[('Q'+i+'E') for i in questions]].sum(axis=1)
  data['delay'] = data['delay'] ** (1/10)
  data['delay_var'] = data['delay'].var()

In [34]:

Ancoms = list(combinations(answers, 2))
for data in dataset:
  for a,b in Ancoms:
    data['mach_%s_dv_%s'%(a,b)] = data[a]/data[b]

In [35]:
for data in dataset:
  data['mach_var'] = data[answers].var(axis=1)


### 나머지 Features


In [36]:
tps = ['tp01', 'tp02', 'tp03', 'tp04', 'tp05', 'tp06', 'tp07', 'tp08', 'tp09', 'tp10']
for data in dataset:
  for tp in tps:
    data[tp] = 7 - data[tp]

tipi feature들을 일반적인 형태로 복구시켜줬습니다.

In [37]:
for data in dataset:
  for tp in tps:
    data[tp] = data[tp].replace(0, np.nan)
    mean = data[tp].mean(axis=0)
    data[tp] = data[tp].replace(np.nan , mean)


tp중 무응답 값들을 평균값으로 대체했습니다.

In [38]:
for data in dataset:
  data['Ex'] = (data['tp01']+data['tp06'])/2
  data['Ag'] = (data['tp07']+data['tp02'])/2
  data['Con'] = (data['tp03']+data['tp08'])/2
  data['Es'] =(data['tp09']+data['tp04'])/2
  data['Op'] =(data['tp05']+data['tp10'])/2

In [39]:
index = test['index']
for data in dataset:
  data.drop('index', axis = 1, inplace = True)

In [40]:
import numpy as np
for data in dataset:
  teenager_ox = 1*np.array(data['age_group'] == '10s')
  data['teenager_ox'] = teenager_ox

10대인지 아닌지의 여부가 투표 여부에 큰 영향을 미칠 것 같아 하나의 column을 더 만들어주었습니다. 

In [41]:
tpcoms = list(combinations(tps, 2))
for data in dataset:
  for a,b in tpcoms:
    data['tp_%s_dv_%s'%(a,b)] = data[a]/data[b]

tp 값들끼리 나눈 feature들을 생성해주었습니다.

In [42]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
needenco = ['age_group', 'gender', 'race', 'religion']
for i in needenco:
  x_train[i] = encoder.fit_transform(x_train[i])
  test[i] = encoder.transform(test[i])

In [43]:
for data in dataset:
  data['Es_gender'] = data['Es']*data['gender']
  data['Con_gender'] = data['Con']*data['gender']
  data['Op_gender'] = data['Op']*data['gender']

EDA 결과, 성별에 따라 Emotional Stability/ Conscience/ Open Minded가 투표 여부에 미치는 영향이 크다고 판단되어 feature를 추가해주었습니다.

정보 출처: https://www.sciencedirect.com/science/article/abs/pii/S0261379413001613

## 4. Feature Selection 1 & Model 2-1

In [45]:
def lgbm_rfe_4040(x_data, y_data, ratio=0.9, min_feats=40):
    feats = x_data.columns.tolist()
    archive = pd.DataFrame(columns=['model', 'n_feats', 'feats', 'score'])
    while True:
        model = LGBMClassifier(objective = 'binary', num_iterations=10**4)
        x_train, x_val, y_train, y_val = train_test_split(x_data[feats], y_data, random_state=4040)
        model.fit(x_train, y_train, eval_set=[(x_val, y_val)], early_stopping_rounds=100, verbose=False)
        val_pred = model.predict_proba(x_val)
        val_pred = val_pred[:,1]
        score = roc_auc_score(y_val, val_pred)
        n_feats = len(feats)
        print(n_feats, score)
        archive = archive.append({'model': model, 'n_feats': n_feats, 'feats': feats, 'score': score}, ignore_index=True)
        feat_imp = pd.Series(model.feature_importances_, index=feats).sort_values(ascending=False)        
        next_n_feats = int(n_feats * ratio)
        if next_n_feats < min_feats:
            break
        else:
            feats = feat_imp.iloc[:next_n_feats].index.tolist()
    return archive


In [46]:
lgbm_archive_4040 = lgbm_rfe_4040(x_train, y_train)

328 0.7667590988557663
295 0.7667590988557663
265 0.7664205423454967
238 0.76621676206039
214 0.766182112419842
192 0.7660918058230415
172 0.7667207511818324
154 0.7662922454477363
138 0.7666096858766138
124 0.7669425399572507
111 0.766992945084577
99 0.7667622064468469
89 0.7667038925002204
80 0.7661617266223537
72 0.7658546655476854
64 0.7667790185145925
57 0.7672032513109528
51 0.7676086054914988
45 0.7666853867953358
40 0.7660839436176077


In [78]:
lgbm_archive_4040

,model,n_feats,feats,score
0,"LGBMClassifier(num_iterations=10000, objective...",328,"[QaA, QaE, QbA, QbE, QcA, QcE, QdA, QdE, QeA, ...",0.766759
1,"LGBMClassifier(num_iterations=10000, objective...",295,"[education, race, religion, Ag_education, marr...",0.766759
2,"LGBMClassifier(num_iterations=10000, objective...",265,"[education, race, religion, Ag_education, marr...",0.766421
3,"LGBMClassifier(num_iterations=10000, objective...",238,"[education, race, religion, Ag_education, marr...",0.766217
4,"LGBMClassifier(num_iterations=10000, objective...",214,"[education, race, religion, Ag_education, marr...",0.766182
5,"LGBMClassifier(num_iterations=10000, objective...",192,"[education, race, religion, married, teenager_...",0.766092
6,"LGBMClassifier(num_iterations=10000, objective...",172,"[education, race, religion, Ag_education, marr...",0.766721
7,"LGBMClassifier(num_iterations=10000, objective...",154,"[education, race, religion, Ag_education, marr...",0.766292
8,"LGBMClassifier(num_iterations=10000, objective...",138,"[education, race, religion, Ag_education, marr...",0.766610
9,"LGBMClassifier(num_iterations=10000, objective...",124,"[education, race, religion, Ag_education, marr...",0.766943


In [80]:
lgbm_archive_4040[lgbm_archive_4040['score']==lgbm_archive_4040['score'].max()].index[0]

17

In [47]:
model = LGBMClassifier(objective="binary", num_iterations= 10**3)

x_train_1 = x_train[lgbm_archive_4040.iloc[lgbm_archive_4040[lgbm_archive_4040['score']==lgbm_archive_4040['score'].max()].index[0],2]]

model.fit(x_train_1, y_train)

pred_y1 = model.predict_proba(test[lgbm_archive_4040.iloc[lgbm_archive_4040[lgbm_archive_4040['score']==lgbm_archive_4040['score'].max()].index[0],2]])
pred_y1 = pred_y1[:,1]

## 5. Feature Selection 2 & Model 2-2

In [48]:
def lgbm_rfe_1234(x_data, y_data, ratio=0.9, min_feats=40):
    feats = x_data.columns.tolist()
    archive = pd.DataFrame(columns=['model', 'n_feats', 'feats', 'score'])
    while True:
        model = LGBMClassifier(objective = 'binary', num_iterations=10**4)
        x_train, x_val, y_train, y_val = train_test_split(x_data[feats], y_data, random_state=1234)
        model.fit(x_train, y_train, eval_set=[(x_val, y_val)], early_stopping_rounds=100, verbose=False)
        val_pred = model.predict_proba(x_val)
        val_pred = val_pred[:,1]
        score = roc_auc_score(y_val, val_pred)
        n_feats = len(feats)
        print(n_feats, score)
        archive = archive.append({'model': model, 'n_feats': n_feats, 'feats': feats, 'score': score}, ignore_index=True)
        feat_imp = pd.Series(model.feature_importances_, index=feats).sort_values(ascending=False)
        next_n_feats = int(n_feats * ratio)
        if next_n_feats < min_feats:
            break
        else:
            feats = feat_imp.iloc[:next_n_feats].index.tolist()
    return archive


In [49]:
lgbm_archive_1234 = lgbm_rfe_1234(x_train, y_train)

328 0.7605179231866099
295 0.7605598788238965
265 0.7605598788238965
238 0.7602440902354561
214 0.7607388833134925
192 0.7610624276210749
172 0.760490497842415
154 0.7604886601418954
138 0.7603601301216558
124 0.7606148385284193
111 0.7619022256110666
99 0.7607998233315704
89 0.7607141989463435
80 0.7606005418413261
72 0.7602810311306467
64 0.7598087109496428
57 0.7596328523541567
51 0.7584910952508211
45 0.7592331680648764
40 0.7587195774908473


In [50]:
model2 = LGBMClassifier(objective="binary", num_iterations= 10**3)

x_train_2 = x_train[lgbm_archive_1234.iloc[lgbm_archive_1234[lgbm_archive_1234['score']==lgbm_archive_1234['score'].max()].index[0],2]]

model2.fit(x_train_2, y_train)

pred_y2 = model2.predict_proba(test[lgbm_archive_1234.iloc[lgbm_archive_1234[lgbm_archive_1234['score']==lgbm_archive_1234['score'].max()].index[0],2]])
pred_y2 = pred_y2[:,1]

## 6. Feature Selection 3 & Model 2-3

In [51]:
def lgbm_rfe_99087(x_data, y_data, ratio=0.9, min_feats=40):
    feats = x_data.columns.tolist()
    archive = pd.DataFrame(columns=['model', 'n_feats', 'feats', 'score'])
    while True:
        model = LGBMClassifier(objective = 'binary', num_iterations=10**4)
        x_train, x_val, y_train, y_val = train_test_split(x_data[feats], y_data, random_state=99087)
        model.fit(x_train, y_train, eval_set=[(x_val, y_val)], early_stopping_rounds=100, verbose=False)
        val_pred = model.predict_proba(x_val)
        val_pred = val_pred[:,1]
        score = roc_auc_score(y_val, val_pred)
        n_feats = len(feats)
        print(n_feats, score)
        archive = archive.append({'model': model, 'n_feats': n_feats, 'feats': feats, 'score': score}, ignore_index=True)
        feat_imp = pd.Series(model.feature_importances_, index=feats).sort_values(ascending=False)
        next_n_feats = int(n_feats * ratio)
        if next_n_feats < min_feats:
            break
        else:
            feats = feat_imp.iloc[:next_n_feats].index.tolist()
    return archive


In [52]:
lgbm_archive_99087 = lgbm_rfe_99087(x_train, y_train)

328 0.7610747678039972
295 0.7610747678039972
265 0.7604277606417176
238 0.7602774024530238
214 0.7598750464304842
192 0.7597839598755943
172 0.760346417795536
154 0.7595337053250955
138 0.7605244194773064
124 0.7592047072884062
111 0.7601848527853248
99 0.7600930969341498
89 0.7592065283969027
80 0.7604775998674109
72 0.7602833949895298
64 0.7590251802337071
57 0.7582892811857515
51 0.7593262234767222
45 0.7602728107692075
40 0.7609816422301925


In [53]:
model3 = LGBMClassifier(objective="binary", num_iterations= 10**3)

x_train_3 = x_train[lgbm_archive_99087.iloc[lgbm_archive_99087[lgbm_archive_99087['score']==lgbm_archive_99087['score'].max()].index[0],2]]

model3.fit(x_train_3, y_train)

pred_y3 = model3.predict_proba(test[lgbm_archive_99087.iloc[lgbm_archive_99087[lgbm_archive_99087['score']==lgbm_archive_99087['score'].max()].index[0],2]])
pred_y3 = pred_y3[:,1]

## 7. Feature Selection 4 & Model 2-4 

In [54]:
def lgbm_rfe_42(x_data, y_data, ratio=0.9, min_feats=40):
    feats = x_data.columns.tolist()
    archive = pd.DataFrame(columns=['model', 'n_feats', 'feats', 'score'])
    while True:
        model = LGBMClassifier(objective = 'binary', num_iterations=10**4)
        x_train, x_val, y_train, y_val = train_test_split(x_data[feats], y_data, random_state=42)
        model.fit(x_train, y_train, eval_set=[(x_val, y_val)], early_stopping_rounds=100, verbose=False)
        val_pred = model.predict_proba(x_val)
        val_pred = val_pred[:,1]
        score = roc_auc_score(y_val, val_pred)
        n_feats = len(feats)
        print(n_feats, score)
        archive = archive.append({'model': model, 'n_feats': n_feats, 'feats': feats, 'score': score}, ignore_index=True)
        feat_imp = pd.Series(model.feature_importances_, index=feats).sort_values(ascending=False)
        next_n_feats = int(n_feats * ratio)
        if next_n_feats < min_feats:
            break
        else:
            feats = feat_imp.iloc[:next_n_feats].index.tolist()
    return archive


In [55]:
lgbm_archive_42 = lgbm_rfe_42(x_train, y_train)

328 0.7702016191143759
295 0.7701378938189016
265 0.7704476479419937
238 0.7696175224576409
214 0.7692701620048307
192 0.7708429035423793
172 0.767934888251297
154 0.7698726104259439
138 0.7697194767041864
124 0.7703372727416495
111 0.7688017016988721
99 0.76799360144488
89 0.7693216216596519
80 0.7679889940468683
72 0.768751829728501
64 0.7684515705810414
57 0.769138570981885
51 0.7703638586734205
45 0.7695835895939052
40 0.7689589135904917


In [56]:
model4 = LGBMClassifier(objective="binary", num_iterations= 10**3)

x_train_4 = x_train[lgbm_archive_42.iloc[lgbm_archive_42[lgbm_archive_42['score']==lgbm_archive_42['score'].max()].index[0],2]]

model4.fit(x_train_4, y_train)

pred_y4 = model4.predict_proba(test[lgbm_archive_42.iloc[lgbm_archive_42[lgbm_archive_42['score']==lgbm_archive_42['score'].max()].index[0],2]])
pred_y4 = pred_y4[:,1]

## 8. Ensemble

In [57]:
pred_all = (pred_y + pred_y2 + pred_y3 + pred_y4) * (1/4)

submission = pd.DataFrame({
    "index" : index,
    "voted" : pred_all
})
submission.to_csv('./data/model2_2.csv', index=False)


# **Model3: NN**

3번째 모델은 Junho Sun 님께서 공유해주신 코드를 그대로 활용하였습니다.

좋은 모델을 공유해주신 덕분에 public score도 0.78대로 올라갈 수 있었습니다. 
정말 감사합니다!

In [58]:
import random
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import StratifiedKFold
from torch import nn, optim
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset
from tqdm import tqdm

random.seed(0)
np.random.seed(0)
torch.manual_seed(0)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

drop_list = ['QaE', 'QbE', 'QcE', 'QdE', 'QeE',
             'QfE', 'QgE', 'QhE', 'QiE', 'QjE',
             'QkE', 'QlE', 'QmE', 'QnE', 'QoE',
             'QpE', 'QqE', 'QrE', 'QsE', 'QtE',
             'index', 'hand']
replace_dict = {'education': str, 'engnat': str, 'married': str, 'urban': str}
train_data = pd.read_csv('./data/train.csv')
test_data = pd.read_csv('./data/test_x.csv')
train_data = train_data.drop(train_data[train_data.familysize > 50].index)
train_y = train_data['voted']
train_x = train_data.drop(drop_list + ['voted'], axis=1)
test_x = test_data.drop(drop_list, axis=1)
train_x = train_x.astype(replace_dict)
test_x = test_x.astype(replace_dict)
train_x = pd.get_dummies(train_x)
test_x = pd.get_dummies(test_x)
train_y = 2 - train_y.to_numpy()
train_x = train_x.to_numpy()
test_x = test_x.to_numpy()

train_y_t = torch.tensor(train_y, dtype=torch.float32)
train_x_t = torch.tensor(train_x, dtype=torch.float32)
test_x_t = torch.tensor(test_x, dtype=torch.float32)
train_x_t[:, :20] = (train_x_t[:, :20] - 3.) / 2.
test_x_t[:, :20] = (test_x_t[:, :20] - 3.) / 2
train_x_t[:, 20] = (train_x_t[:, 20] - 5.) / 4.
test_x_t[:, 20] = (test_x_t[:, 20] - 5.) / 4.
train_x_t[:, 21:31] = (train_x_t[:, 21:31] - 3.5) / 3.5
test_x_t[:, 21:31] = (test_x_t[:, 21:31] - 3.5) / 3.5
test_len = len(test_x_t)

N_REPEAT = 10
N_SKFOLD = 7
N_EPOCH = 48
BATCH_SIZE = 72
LOADER_PARAM = {
    'batch_size': BATCH_SIZE,
    'num_workers': 4,
    'pin_memory': True
}
prediction = np.zeros((test_len, 1), dtype=np.float32)

for repeat in range(N_REPEAT):

    skf, tot = StratifiedKFold(n_splits=N_SKFOLD, random_state=repeat, shuffle=True), 0.
    for skfold, (train_idx, valid_idx) in enumerate(skf.split(train_x, train_y)):
        train_idx, valid_idx = list(train_idx), list(valid_idx)
        train_loader = DataLoader(TensorDataset(train_x_t[train_idx, :], train_y_t[train_idx]),
                                  shuffle=True, drop_last=True, **LOADER_PARAM)
        valid_loader = DataLoader(TensorDataset(train_x_t[valid_idx, :], train_y_t[valid_idx]),
                                  shuffle=False, drop_last=False, **LOADER_PARAM)
        test_loader = DataLoader(TensorDataset(test_x_t, torch.zeros((test_len,), dtype=torch.float32)),
                                 shuffle=False, drop_last=False, **LOADER_PARAM)
        model = nn.Sequential(
            nn.Dropout(0.05),
            nn.Linear(91, 180, bias=False),
            nn.LeakyReLU(0.05, inplace=True),
            nn.Dropout(0.5),
            nn.Linear(180, 32, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(32, 1)
        ).to(DEVICE)
        criterion = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor([1.20665], device=DEVICE))
        optimizer = optim.AdamW(model.parameters(), lr=5e-3, weight_decay=7.8e-2)
        scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=N_EPOCH // 6, eta_min=4e-4)
        prediction_t, loss_t = np.zeros((test_len, 1), dtype=np.float32), 1.

        # for epoch in range(N_EPOCH):
        for epoch in tqdm(range(N_EPOCH), desc='{:02d}/{:02d}'.format(skfold + 1, N_SKFOLD)):
            model.train()
            for idx, (xx, yy) in enumerate(train_loader):
                optimizer.zero_grad()
                xx, yy = xx.to(DEVICE), yy.to(DEVICE)
                pred = model(xx).squeeze()
                loss = criterion(pred, yy)
                loss.backward()
                optimizer.step()
                scheduler.step(epoch + idx / len(train_loader))

            with torch.no_grad():
                model.eval()
                running_acc, running_loss, running_count = 0, 0., 0
                for xx, yy in valid_loader:
                    xx, yy = xx.to(DEVICE), yy.to(DEVICE)
                    pred = model(xx).squeeze()
                    loss = criterion(pred, yy)
                    running_loss += loss.item() * len(yy)
                    running_count += len(yy)
                    running_acc += ((torch.sigmoid(pred) > 0.5).float() == yy).sum().item()
                # print('R{:02d} S{:02d} E{:02d} | {:6.4f}, {:5.2f}%'
                #       .format(repeat + 1, skfold + 1, epoch + 1, running_loss / running_count,
                #               running_acc / running_count * 100))

                if running_loss / running_count < loss_t:
                    loss_t = running_loss / running_count
                    for idx, (xx, _) in enumerate(test_loader):
                        xx = xx.to(DEVICE)
                        pred = (2. - torch.sigmoid(model(xx).detach().to('cpu'))).numpy()
                        prediction_t[BATCH_SIZE * idx:min(BATCH_SIZE * (idx + 1), len(prediction)), :] \
                            = pred[:, :].copy()
        prediction[:, :] += prediction_t[:, :].copy() / (N_REPEAT * N_SKFOLD)
        tot += loss_t
    print('R{} -> {:6.4f}'.format(repeat + 1, tot / N_SKFOLD))

df = pd.read_csv('./data/sample_submission.csv')
df.iloc[:, 1:] = prediction

07/07: 100%|██████████| 48/48 [01:11<00:00,  1.49s/it]


R1 -> 0.6054


07/07: 100%|██████████| 48/48 [01:12<00:00,  1.50s/it]


R2 -> 0.6053


07/07: 100%|██████████| 48/48 [01:12<00:00,  1.52s/it]


...

R8 -> 0.6053


07/07: 100%|██████████| 48/48 [01:08<00:00,  1.44s/it]


R9 -> 0.6051


07/07: 100%|██████████| 48/48 [01:11<00:00,  1.49s/it]

R10 -> 0.6058


In [59]:
df.to_csv('./data/model3_2.csv', index=False)

# Final Ensemble

In [87]:
model1 = pd.read_csv('./data/model1.csv', index_col = 'index')
model2 = pd.read_csv('./data/model2.csv', index_col='index')

pred_y = (model1)*(0.7) + (model2)*(0.3)

test = pd.read_csv('./data/test_x.csv')
index = test['index']

submission = pd.DataFrame({
    'index': index,
    'voted': pred_y['voted']
    })

submission.to_csv('./data/combined_model1_model2.csv', index=False)

In [88]:

combined_12 = pd.read_csv('./data/combined_model1_model2.csv', index_col = 'index')
model3 = pd.read_csv('./data/model3.csv', index_col='index')
model3['voted'] = model3['voted']-1

다른 모델과 같이 [0,1]의 범위(voted가 2일 확률)를 맞춰주기 위해 1을 빼주었습니다.

In [85]:
pred_y = (model3)*(0.8) + (combined_12)*(0.2)

test = pd.read_csv('./data/test_x.csv')
index = test['index']

submission = pd.DataFrame({
    'index': index,
    'voted': pred_y['voted']
    })

submission.to_csv('./data/submission_final_3_copy.csv', index=False)